In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = Path(
    r"C:\Users\Moni\uncertainty-aware-rul-prediction"
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_split = pd.read_csv(PROCESSED_DIR / "train_split.csv")
val_split = pd.read_csv(PROCESSED_DIR / "validation_split.csv")

# Use sensor readings and operating settings only
EXCLUDE = ["RUL", "engine_id", "cycle"]
FEATURE_COLS = [
    c for c in train_split.columns if c not in EXCLUDE
]

X_train = train_split[FEATURE_COLS]
y_train = train_split["RUL"]

X_val = val_split[FEATURE_COLS]
y_val = val_split["RUL"]

# Train lower and upper quantile models
lower_model = GradientBoostingRegressor(
    loss="quantile",
    alpha=0.10,
    n_estimators=100,
    max_depth=3,
    random_state=42
)

upper_model = GradientBoostingRegressor(
    loss="quantile",
    alpha=0.90,
    n_estimators=100,
    max_depth=3,
    random_state=42
)

lower_model.fit(X_train, y_train)
upper_model.fit(X_train, y_train)

lower = lower_model.predict(X_val)
upper = upper_model.predict(X_val)

# Ensure valid interval ordering
lower_bound = np.minimum(lower, upper)
upper_bound = np.maximum(lower, upper)

print("Lower-bound predictions:", lower_bound[:5].round(1))
print("Upper-bound predictions:", upper_bound[:5].round(1))
print("Validation samples:", len(y_val))

Lower-bound predictions: [90.9 87.8 91.5 99.6 81.7]
Upper-bound predictions: [228.9 221.1 212.  228.8 206.4]
Validation samples: 9498


In [2]:
import numpy as np
import pandas as pd

# Actual RUL values
y_true = y_val.to_numpy()

# Check interval quality
coverage = np.mean(
    (y_true >= lower_bound) & (y_true <= upper_bound)
)

avg_width = np.mean(upper_bound - lower_bound)
median_width = np.median(upper_bound - lower_bound)

# Quantile regression loss
def pinball_loss(y_true, y_pred, quantile):
    errors = y_true - y_pred
    return np.mean(
        np.maximum(
            quantile * errors,
            (quantile - 1) * errors
        )
    )

lower_loss = pinball_loss(y_true, lower_bound, 0.10)
upper_loss = pinball_loss(y_true, upper_bound, 0.90)

print(f"Target coverage: 80%")
print(f"Actual coverage: {coverage:.2%}")
print(f"Average interval width: {avg_width:.2f} cycles")
print(f"Median interval width: {median_width:.2f} cycles")
print(f"Lower quantile pinball loss: {lower_loss:.3f}")
print(f"Upper quantile pinball loss: {upper_loss:.3f}")

Target coverage: 80%
Actual coverage: 78.09%
Average interval width: 107.17 cycles
Median interval width: 123.77 cycles
Lower quantile pinball loss: 6.167
Upper quantile pinball loss: 8.580


In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingRegressor

# Split by engine to avoid leakage between time-series observations
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)

fit_idx, cal_idx = next(
    splitter.split(
        train_split,
        groups=train_split["engine_id"]
    )
)

fit_df = train_split.iloc[fit_idx]
cal_df = train_split.iloc[cal_idx]

X_fit = fit_df[FEATURE_COLS]
y_fit = fit_df["RUL"].to_numpy()

X_cal = cal_df[FEATURE_COLS]
y_cal = cal_df["RUL"].to_numpy()

# Retrain quantile models using only the fitting engines
lower_model = GradientBoostingRegressor(
    loss="quantile", alpha=0.10,
    n_estimators=100, max_depth=3, random_state=42
)
upper_model = GradientBoostingRegressor(
    loss="quantile", alpha=0.90,
    n_estimators=100, max_depth=3, random_state=42
)

lower_model.fit(X_fit, y_fit)
upper_model.fit(X_fit, y_fit)

# Calculate calibration scores
cal_lower = lower_model.predict(X_cal)
cal_upper = upper_model.predict(X_cal)

cal_lo = np.minimum(cal_lower, cal_upper)
cal_hi = np.maximum(cal_lower, cal_upper)

# Conformal score: distance outside the predicted interval
scores = np.maximum(cal_lo - y_cal, y_cal - cal_hi)

alpha = 0.20
n_cal = len(scores)
q_level = min(1.0, np.ceil((n_cal + 1) * (1 - alpha)) / n_cal)
adjustment = np.quantile(scores, q_level, method="higher")

# Evaluate on held-out validation engines
val_lower = lower_model.predict(X_val)
val_upper = upper_model.predict(X_val)

val_lo = np.maximum(
    0, np.minimum(val_lower, val_upper) - adjustment
)
val_hi = np.maximum(val_lower, val_upper) + adjustment

y_true = y_val.to_numpy()

coverage = np.mean((y_true >= val_lo) & (y_true <= val_hi))
avg_width = np.mean(val_hi - val_lo)

print("Fit engines:", fit_df["engine_id"].nunique())
print("Calibration engines:", cal_df["engine_id"].nunique())
print(f"Conformal adjustment: {adjustment:.2f} cycles")
print(f"Calibrated validation coverage: {coverage:.2%}")
print(f"Average calibrated interval width: {avg_width:.2f} cycles")

Fit engines: 139
Calibration engines: 35
Conformal adjustment: 0.87 cycles
Calibrated validation coverage: 78.91%
Average calibrated interval width: 107.77 cycles


In [4]:
print("Calibration score quantiles:")
print(np.quantile(scores, [0, 0.5, 0.8, 0.9, 0.95, 1.0]).round(3))

print("\nCalibration score summary:")
print(pd.Series(scores).describe().round(3))

print("\nValidation engines:", val_split["engine_id"].nunique())
print("Calibration engines:", cal_df["engine_id"].nunique())

print("\nInvalid intervals:", int(np.sum(val_lo > val_hi)))

Calibration score quantiles:
[-73.557 -17.251   0.862  13.163  30.701 118.597]

Calibration score summary:
count    7472.000
mean      -18.050
std        27.331
min       -73.557
25%       -38.464
50%       -17.251
75%        -2.805
max       118.597
dtype: float64

Validation engines: 44
Calibration engines: 35

Invalid intervals: 0


In [5]:
import json
import joblib
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(lower_model, MODEL_DIR / "rul_lower_quantile.joblib")
joblib.dump(upper_model, MODEL_DIR / "rul_upper_quantile.joblib")

config = {
    "model_type": "gradient_boosting_quantile",
    "lower_quantile": 0.10,
    "upper_quantile": 0.90,
    "target_coverage": 0.80,
    "conformal_adjustment": float(adjustment),
    "feature_columns": FEATURE_COLS,
    "target": "RUL"
}

with open(MODEL_DIR / "uncertainty_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Saved calibrated quantile models and configuration.")

Saved calibrated quantile models and configuration.


In [6]:
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "models"

files = [
    "rul_lower_quantile.joblib",
    "rul_upper_quantile.joblib",
    "uncertainty_config.json",
]

for file in files:
    path = MODEL_DIR / file
    print(f"{file}: {'OK' if path.exists() else 'MISSING'}")

rul_lower_quantile.joblib: OK
rul_upper_quantile.joblib: OK
uncertainty_config.json: OK


In [7]:
import json

config_path = PROJECT_ROOT / "models" / "uncertainty_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=4))

{
    "model_type": "gradient_boosting_quantile",
    "lower_quantile": 0.1,
    "upper_quantile": 0.9,
    "target_coverage": 0.8,
    "conformal_adjustment": 0.8743710618477962,
    "feature_columns": [
        "setting_1",
        "setting_2",
        "setting_3",
        "sensor_1",
        "sensor_2",
        "sensor_3",
        "sensor_4",
        "sensor_5",
        "sensor_6",
        "sensor_7",
        "sensor_8",
        "sensor_9",
        "sensor_10",
        "sensor_11",
        "sensor_12",
        "sensor_13",
        "sensor_14",
        "sensor_15",
        "sensor_16",
        "sensor_17",
        "sensor_18",
        "sensor_19",
        "sensor_20",
        "sensor_21"
    ],
    "target": "RUL"
}


In [8]:
import pandas as pd
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "data" / "processed"

val_df = pd.read_csv(DATA_DIR / "validation_split.csv")

print("Validation rows:", len(val_df))
print("Validation engines:", val_df["engine_id"].nunique())
print("Missing values:", val_df.isnull().sum().sum())
print("RUL range:", val_df["RUL"].min(), "to", val_df["RUL"].max())

Validation rows: 9498
Validation engines: 44
Missing values: 0
RUL range: 0 to 283


In [9]:
import numpy as np
import pandas as pd
import joblib

MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

val_df = pd.read_csv(DATA_DIR / "validation_split.csv")

lower_model = joblib.load(MODEL_DIR / "rul_lower_quantile.joblib")
upper_model = joblib.load(MODEL_DIR / "rul_upper_quantile.joblib")

FEATURE_COLS = config["feature_columns"]
X_val = val_df[FEATURE_COLS]
y_val = val_df["RUL"].to_numpy()

lower = lower_model.predict(X_val)
upper = upper_model.predict(X_val)

lower_bound = np.maximum(
    0, np.minimum(lower, upper) - config["conformal_adjustment"]
)
upper_bound = np.maximum(lower, upper) + config["conformal_adjustment"]

results = val_df[["engine_id", "RUL"]].copy()
results["lower_bound"] = lower_bound
results["upper_bound"] = upper_bound
results["covered"] = (
    (results["RUL"] >= results["lower_bound"]) &
    (results["RUL"] <= results["upper_bound"])
)
results["interval_width"] = (
    results["upper_bound"] - results["lower_bound"]
)

engine_results = results.groupby("engine_id").agg(
    coverage=("covered", "mean"),
    mean_interval_width=("interval_width", "mean"),
    rows=("covered", "size")
)

print("Per-engine coverage summary:")
print(engine_results["coverage"].describe().round(3))

print("\nLowest-coverage engines:")
print(engine_results.sort_values("coverage").head(5).round(3))

print("\nHighest-coverage engines:")
print(engine_results.sort_values("coverage", ascending=False).head(5).round(3))

Per-engine coverage summary:
count    44.000
mean      0.793
std       0.101
min       0.575
25%       0.709
50%       0.797
75%       0.882
max       0.948
Name: coverage, dtype: float64

Lowest-coverage engines:
           coverage  mean_interval_width  rows
engine_id                                     
109           0.575              110.324   160
56            0.630              119.469   162
114           0.640              121.381   197
212           0.646              107.387   277
80            0.651              105.651   284

Highest-coverage engines:
           coverage  mean_interval_width  rows
engine_id                                     
74            0.948              109.810   192
190           0.941              106.191   152
10            0.934              101.557   228
193           0.933              116.021   223
179           0.919              114.861   223


In [10]:
results["rul_group"] = pd.cut(
    results["RUL"],
    bins=[-1, 30, 60, 100, 150, float("inf")],
    labels=["0–30", "31–60", "61–100", "101–150", "150+"]
)

print(
    results.groupby("rul_group", observed=True)
    .agg(
        coverage=("covered", "mean"),
        mean_interval_width=("interval_width", "mean"),
        samples=("covered", "size")
    )
    .round(3)
)

           coverage  mean_interval_width  samples
rul_group                                        
0–30          0.795               50.526     1364
31–60         0.688               82.627     1320
61–100        0.856              111.980     1760
101–150       0.981              125.552     2170
150+          0.647              130.416     2884


In [11]:
engine_group_counts = (
    results.groupby("rul_group", observed=True)["engine_id"]
    .nunique()
)

print("Number of engines represented in each RUL range:")
print(engine_group_counts)

Number of engines represented in each RUL range:
rul_group
0–30       44
31–60      44
61–100     44
101–150    44
150+       41
Name: engine_id, dtype: int64


In [13]:
import numpy as np
import pandas as pd
import joblib
import json
import torch

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch import nn

MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(DATA_DIR / "train_split.csv")
val_df = pd.read_csv(DATA_DIR / "validation_split.csv")

with open(MODEL_DIR / "model_config.json") as f:
    model_config = json.load(f)

with open(MODEL_DIR / "uncertainty_config.json") as f:
    uncertainty_config = json.load(f)

FEATURE_COLS = uncertainty_config["feature_columns"]
WINDOW_SIZE = model_config["window_size"]

# Same validation rows with complete 30-cycle windows
val_df = val_df.sort_values(["engine_id", "cycle"]).copy()
val_df["window_count"] = val_df.groupby("engine_id").cumcount() + 1
eval_df = val_df[val_df["window_count"] >= WINDOW_SIZE].copy()

X_train = train_df[FEATURE_COLS]
X_eval = eval_df[FEATURE_COLS]
y_eval = eval_df["RUL"].to_numpy()

# 1. Random Forest baseline
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, train_df["RUL"])
rf_pred = rf.predict(X_eval)

# 2. Quantile-model midpoint
lower_model = joblib.load(MODEL_DIR / "rul_lower_quantile.joblib")
upper_model = joblib.load(MODEL_DIR / "rul_upper_quantile.joblib")

q_lower = lower_model.predict(X_eval)
q_upper = upper_model.predict(X_eval)
quantile_pred = (q_lower + q_upper) / 2

# 3. Load the saved CNN
class RULCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

    

scaler = joblib.load(MODEL_DIR / "feature_scaler.joblib")
cnn = RULCNN(len(FEATURE_COLS))
state = torch.load(
    MODEL_DIR / "rul_cnn.pt",
    map_location="cpu",
    weights_only=True
)
cnn.load_state_dict(state)
cnn.eval()

# Build windows independently for each engine
scaled_val = val_df.copy()
scaled_val[FEATURE_COLS] = scaler.transform(
    scaled_val[FEATURE_COLS]
)

X_windows = []
y_windows = []
eval_indices = []

for _, group in scaled_val.groupby("engine_id", sort=False):
    group = group.sort_values("cycle")
    features = group[FEATURE_COLS].to_numpy(dtype=np.float32)
    targets = group["RUL"].to_numpy()

    for end in range(WINDOW_SIZE - 1, len(group)):
        X_windows.append(features[end - WINDOW_SIZE + 1:end + 1])
        y_windows.append(targets[end])
        eval_indices.append(group.index[end])

X_windows = np.asarray(X_windows, dtype=np.float32)

with torch.no_grad():
    cnn_pred = cnn(
        torch.tensor(X_windows).transpose(1, 2)
    ).squeeze(1).numpy()

# Align all predictions to the same validation rows
cnn_results = pd.DataFrame({
    "index": eval_indices,
    "CNN": cnn_pred,
}).set_index("index")

comparison = eval_df.join(cnn_results, how="inner")
comparison["Random Forest"] = rf_pred[
    eval_df.index.get_indexer(comparison.index)
]
comparison["Quantile midpoint"] = quantile_pred[
    eval_df.index.get_indexer(comparison.index)
]

y = comparison["RUL"].to_numpy()

for name in ["Random Forest", "CNN", "Quantile midpoint"]:
    pred = comparison[name].to_numpy()
    print(
        f"{name}: "
        f"MAE={mean_absolute_error(y, pred):.3f}, "
        f"RMSE={np.sqrt(mean_squared_error(y, pred)):.3f}"
    )

print("Common validation rows:", len(comparison))

Random Forest: MAE=26.801, RMSE=35.695
CNN: MAE=28.165, RMSE=37.936
Quantile midpoint: MAE=32.050, RMSE=40.434
Common validation rows: 8222


In [14]:
# Evaluate Random Forest errors by RUL range

comparison["rf_prediction"] = rf_pred[
    eval_df.index.get_indexer(comparison.index)
]

comparison["absolute_error"] = np.abs(
    comparison["RUL"] - comparison["rf_prediction"]
)

comparison["rul_group"] = pd.cut(
    comparison["RUL"],
    bins=[-1, 30, 60, 100, 150, float("inf")],
    labels=["0–30", "31–60", "61–100", "101–150", "150+"]
)

rf_error_by_range = comparison.groupby(
    "rul_group", observed=True
).agg(
    MAE=("absolute_error", "mean"),
    samples=("absolute_error", "size")
)

print(rf_error_by_range.round(3))

              MAE  samples
rul_group                 
0–30        8.690     1364
31–60      24.027     1320
61–100     32.129     1758
101–150    24.949     1954
150+       39.188     1826


In [15]:
comparison["signed_error"] = (
    comparison["rf_prediction"] - comparison["RUL"]
)

bias_by_range = comparison.groupby(
    "rul_group", observed=True
).agg(
    Mean_signed_error=("signed_error", "mean"),
    MAE=("absolute_error", "mean"),
    samples=("signed_error", "size")
)

print(bias_by_range.round(3))

           Mean_signed_error     MAE  samples
rul_group                                    
0–30                   6.441   8.690     1364
31–60                 18.221  24.027     1320
61–100                24.402  32.129     1758
101–150               12.039  24.949     1954
150+                 -35.391  39.188     1826


In [16]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = train_df[FEATURE_COLS]
y_train = train_df["RUL"]
groups = train_df["engine_id"]

rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_distributions = {
    "n_estimators": [150, 250, 350],
    "max_depth": [None, 12, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [0.7, 1.0],
    "min_samples_split": [2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=5),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train, groups=groups)

tuned_rf = search.best_estimator_
tuned_pred = tuned_rf.predict(X_eval)

print("Best parameters:")
print(search.best_params_)

print(f"\nGrouped CV MAE: {-search.best_score_:.3f}")
print(f"Validation MAE: {mean_absolute_error(y_eval, tuned_pred):.3f}")
print(
    f"Validation RMSE: "
    f"{np.sqrt(mean_squared_error(y_eval, tuned_pred)):.3f}"
)
print(f"Baseline validation MAE: 26.801")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters:
{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 0.7, 'max_depth': 20}

Grouped CV MAE: 30.447
Validation MAE: 26.274
Validation RMSE: 35.047
Baseline validation MAE: 26.801


In [17]:
import joblib

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(tuned_rf, MODEL_DIR / "rul_random_forest_tuned.joblib")

print("Saved:", MODEL_DIR / "rul_random_forest_tuned.joblib")
print("Validation MAE:", round(26.274, 3))
print("Validation RMSE:", round(35.047, 3))

Saved: C:\Users\Moni\uncertainty-aware-rul-prediction\models\rul_random_forest_tuned.joblib
Validation MAE: 26.274
Validation RMSE: 35.047
